# ProCoders Dataset Developer Burnout

**Resumen de etapas:**
1. Limpieza (duplicados, nulos, rangos)
2. Ingeniería de características
3. Escalado (StandardScaler)
4. Clasificación de perfiles
5. Normalización (MinMaxScaler)
6. Análisis estadístico

In [20]:
# Instalar dependencias
%pip install -q pandas numpy scikit-learn scipy

In [21]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import spearmanr, f_oneway

# Configurar rutas de entrada y salida
ROOT = Path('.').resolve()
INPUT_FILE = 'developer_burnout_dataset_7000.csv'
OUTPUT_FILE = 'dataset_final.csv'

print(f'Directorio de trabajo: {ROOT}')
print(f'Archivo de entrada: {INPUT_FILE}')
print(f'Archivo de salida: {OUTPUT_FILE}')

Directorio de trabajo: /drive/notebooks/ciencias-datos-ev/ciencias-datos-ev
Archivo de entrada: developer_burnout_dataset_7000.csv
Archivo de salida: dataset_final.csv


In [22]:
# Funciones de carga e informe

def load_data(path: str) -> pd.DataFrame:
    """Cargar archivo CSV desde la ruta especificada"""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    return pd.read_csv(path)

def initial_report(df: pd.DataFrame) -> None:
    """Mostrar resumen inicial: tipos, estadísticas, nulos y duplicados"""
    print('\n--- Informe inicial del dataset ---')
    print(f'Forma: {df.shape}')
    print(f'\nTipos de datos:')
    print(df.dtypes)
    print(f'\nDuplicados: {df.duplicated().sum()}')

In [23]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia el dataset: los duplicados, nulos, valores fuera de rango, outliers"""
    
    # Paso 1: Eliminar duplicados exactos
    df = df.drop_duplicates().copy()

    # Paso 2: Imputar nulos en numéricas con mediana
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    medians = df[numeric_cols].median()
    df[numeric_cols] = df[numeric_cols].fillna(medians)

    # Paso 3: Filtrar valores irreales (fuera del rango 0-24 horas)
    if 'sleep_hours' in df.columns:
        df = df[(df['sleep_hours'] >= 0) & (df['sleep_hours'] <= 24)]
    if 'exercise_hours' in df.columns:
        df = df[(df['exercise_hours'] >= 0)]
    if 'daily_work_hours' in df.columns:
        df = df[(df['daily_work_hours'] > 0) & (df['daily_work_hours'] <= 24)]

    # Paso 4: Recortar outliers extremos (1% y 99% percentiles)
    for col in ['stress_level', 'screen_time']:
        if col in df.columns:
            lower = df[col].quantile(0.01)
            upper = df[col].quantile(0.99)
            df[col] = df[col].clip(lower, upper)

    df = df.reset_index(drop=True)
    return df

In [24]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Crea variables derivadas: sleep_quality, workload_index, high_burnout"""
    df = df.copy()
    
    # sleep_quality: penaliza exposición a pantallas alta
    if ('sleep_hours' in df.columns) and ('screen_time' in df.columns):
        df['sleep_quality'] = df['sleep_hours'] / (1 + df['screen_time'])

    # workload_index: carga de trabajo normalizada (0-1)
    workload_cols = [c for c in ['daily_work_hours', 'meetings_per_day', 'commits_per_day'] if c in df.columns]
    if workload_cols:
        raw = df[workload_cols].sum(axis=1)
        df['workload_index'] = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)

    # high_burnout: etiqueta binaria desde burnout_level
    if 'burnout_level' in df.columns:
        df['high_burnout'] = df['burnout_level'].apply(lambda x: 1 if str(x).strip().lower() == 'high' else 0)

    return df

In [25]:
def scale_data(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Aplica StandardScaler (media 0, desv. estándar 1) a columnas seleccionadas"""
    # Crea nuevas columnas con sufijo _scaled, sin sobrescribir originales
    scaled_df = df.copy()
    scaler = StandardScaler()
    cols_present = [c for c in columns if c in df.columns]
    if cols_present:
        scaled_vals = scaler.fit_transform(df[cols_present])
        for i, col in enumerate(cols_present):
            scaled_df[f"{col}_scaled"] = scaled_vals[:, i]
    return scaled_df

In [26]:
def classify_profiles(orig_df: pd.DataFrame, target_df: pd.DataFrame) -> pd.DataFrame:
    """Clasifica en Saludable (sueño>=7, ejercicio>=3) o Riesgo"""
    SLEEP_THRESHOLD = 7.0
    EXERCISE_THRESHOLD = 3.0

    def classify(row):
        has_good_sleep = (row['sleep_hours'] >= SLEEP_THRESHOLD) if 'sleep_hours' in row.index else False
        has_good_exercise = (row['exercise_hours'] >= EXERCISE_THRESHOLD) if 'exercise_hours' in row.index else False
        return 'Saludable' if (has_good_sleep and has_good_exercise) else 'Riesgo'

    target_df['profile_type'] = orig_df.apply(classify, axis=1)
    return target_df

In [27]:
def normalize_and_select(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza a 0-1 con MinMaxScaler y seleccionar columnas finales"""
    df = df.copy()
    
    # Normaliza los valores numéricos (excepto high_burnout que es binaria)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cols_to_norm = [c for c in numeric_cols if c != 'high_burnout']

    if cols_to_norm:
        scaler = MinMaxScaler()
        vals = scaler.fit_transform(df[cols_to_norm])
        for i, col in enumerate(cols_to_norm):
            df[f"{col}_norm"] = vals[:, i]

    # Selecciona las columnas finales: categóricas + binarias + normalizadas
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    keep_cols = cat_cols + ['high_burnout', 'burnout_level', 'profile_type']
    keep_cols += [c for c in df.columns if c.endswith('_norm')]

    # Elimina los duplicados preservando orden
    unique_keep_cols = list(dict.fromkeys([c for c in keep_cols if c in df.columns]))
    return df.loc[:, unique_keep_cols]

In [28]:
# Carga el dataset cochino
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"{INPUT_FILE} no encontrado. Asegúrate de estar en el directorio correcto.")

df_raw = load_data(INPUT_FILE)
print(f'\nDataset cargado: {df_raw.shape}')
initial_report(df_raw)


Dataset cargado: (7000, 12)

--- Informe inicial del dataset ---
Forma: (7000, 12)

Tipos de datos:
age                 float64
experience_years    float64
daily_work_hours    float64
sleep_hours         float64
caffeine_intake     float64
bugs_per_day        float64
commits_per_day     float64
meetings_per_day    float64
screen_time         float64
exercise_hours      float64
stress_level        float64
burnout_level        object
dtype: object

Duplicados: 0


In [29]:
# Estadísticas iniciales: tipos, nulos, descriptivas
print('\nEstadísticas descriptivas (primeras 5 filas):')
display(df_raw.head())

print('\n% de Missing por columna:')
missing_pct = df_raw.isnull().mean() * 100
print(missing_pct[missing_pct > 0].sort_values(ascending=False))


Estadísticas descriptivas (primeras 5 filas):


,age,experience_years,daily_work_hours,sleep_hours,caffeine_intake,bugs_per_day,commits_per_day,meetings_per_day,screen_time,exercise_hours,stress_level,burnout_level
0,26.0,12.0,10.33,4.45,2.0,11.0,4.0,1.0,15.07,0.14,55.96,Medium
1,39.0,10.0,8.62,5.77,5.0,15.0,11.0,5.0,13.25,0.54,82.22,High
2,34.0,13.0,NaN,4.03,5.0,2.0,18.0,9.0,11.18,1.54,61.77,Medium
3,30.0,1.0,6.85,6.47,2.0,15.0,26.0,1.0,11.14,0.96,54.98,Medium
4,27.0,7.0,4.24,5.80,NaN,9.0,17.0,7.0,8.05,0.36,27.90,Low



% de Missing por columna:
age                 2.0
experience_years    2.0
daily_work_hours    2.0
sleep_hours         2.0
caffeine_intake     2.0
bugs_per_day        2.0
commits_per_day     2.0
meetings_per_day    2.0
screen_time         2.0
exercise_hours      2.0
stress_level        2.0
burnout_level       2.0
dtype: float64


In [30]:
# Etapa 1: Limpieza
print('\n=== ETAPA 1: LIMPIEZA DE DATOS ===')
df_clean = clean_data(df_raw)
print(f'Registros: {df_raw.shape[0]} -> {df_clean.shape[0]} (removidos: {df_raw.shape[0] - df_clean.shape[0]})')
print(f'% Missing después: {(df_clean.isnull().mean() * 100).sum():.2f}%')


=== ETAPA 1: LIMPIEZA DE DATOS ===
Registros: 7000 -> 7000 (removidos: 0)
% Missing después: 2.00%


In [31]:
# Etapa 2: Feature Engineering
print('\n=== ETAPA 2: INGENIERÍA DE CARACTERÍSTICAS ===')
df_feat = feature_engineering(df_clean)
print(f'Columnas nuevas: sleep_quality, workload_index, high_burnout')
print(f'Distribución high_burnout: {df_feat["high_burnout"].value_counts().to_dict()}')


=== ETAPA 2: INGENIERÍA DE CARACTERÍSTICAS ===
Columnas nuevas: sleep_quality, workload_index, high_burnout
Distribución high_burnout: {0: 5218, 1: 1782}


In [32]:
# Etapa 3: Escalado
print('\n=== ETAPA 3: ESCALADO CON STANDARDSCALER ===')
cols_to_scale = [
    'age', 'experience_years', 'daily_work_hours', 'sleep_hours', 'caffeine_intake',
    'bugs_per_day', 'commits_per_day', 'meetings_per_day', 'screen_time', 'exercise_hours', 'stress_level'
]
df_scaled = scale_data(df_feat, cols_to_scale)
n_scaled = len([c for c in df_scaled.columns if c.endswith('_scaled')])
print(f'Columnas escaladas (_scaled): {n_scaled}')


=== ETAPA 3: ESCALADO CON STANDARDSCALER ===
Columnas escaladas (_scaled): 11


In [33]:
# Etapa 4: Clasificación de Perfiles
print('\n=== ETAPA 4: CLASIFICACIÓN DE PERFILES ===')
df_prof = classify_profiles(df_feat, df_scaled)
profile_dist = df_prof['profile_type'].value_counts()
print(f'Perfiles creados:')
print(profile_dist)
print(f'Proporción Saludable: {profile_dist.get("Saludable", 0) / len(df_prof) * 100:.1f}%')


=== ETAPA 4: CLASIFICACIÓN DE PERFILES ===
Perfiles creados:
profile_type
Riesgo    7000
Name: count, dtype: int64
Proporción Saludable: 0.0%


In [34]:
# Etapa 5: Normalización y Selección Final
print('\n=== ETAPA 5: NORMALIZACIÓN Y SELECCIÓN FINAL ===')
df_final = normalize_and_select(df_prof)
print(f'Shape final: {df_final.shape}')
print(f'Columnas finales: {df_final.columns.tolist()}')


=== ETAPA 5: NORMALIZACIÓN Y SELECCIÓN FINAL ===
Shape final: (7000, 27)
Columnas finales: ['burnout_level', 'profile_type', 'high_burnout', 'age_norm', 'experience_years_norm', 'daily_work_hours_norm', 'sleep_hours_norm', 'caffeine_intake_norm', 'bugs_per_day_norm', 'commits_per_day_norm', 'meetings_per_day_norm', 'screen_time_norm', 'exercise_hours_norm', 'stress_level_norm', 'sleep_quality_norm', 'workload_index_norm', 'age_scaled_norm', 'experience_years_scaled_norm', 'daily_work_hours_scaled_norm', 'sleep_hours_scaled_norm', 'caffeine_intake_scaled_norm', 'bugs_per_day_scaled_norm', 'commits_per_day_scaled_norm', 'meetings_per_day_scaled_norm', 'screen_time_scaled_norm', 'exercise_hours_scaled_norm', 'stress_level_scaled_norm']


In [35]:
# Etapa 6: Guardado redondeado a 5 decimales
print('\n=== ETAPA 6: GUARDADO DEL DATASET FINAL ===')
# Redondear columnas numéricas a máximo 5 decimales
numeric_cols = df_final.select_dtypes(include=[np.number]).columns
df_final[numeric_cols] = df_final[numeric_cols].round(5)
df_final.to_csv(OUTPUT_FILE, index=False)
print(f'Archivo guardado: {OUTPUT_FILE}')
print(f'\nPrimeras 3 filas del resultado:')
display(df_final.head(3))


=== ETAPA 6: GUARDADO DEL DATASET FINAL ===
Archivo guardado: dataset_final.csv

Primeras 3 filas del resultado:


,burnout_level,profile_type,high_burnout,age_norm,experience_years_norm,daily_work_hours_norm,sleep_hours_norm,caffeine_intake_norm,bugs_per_day_norm,commits_per_day_norm,...,experience_years_scaled_norm,daily_work_hours_scaled_norm,sleep_hours_scaled_norm,caffeine_intake_scaled_norm,bugs_per_day_scaled_norm,commits_per_day_scaled_norm,meetings_per_day_scaled_norm,screen_time_scaled_norm,exercise_hours_scaled_norm,stress_level_scaled_norm
0,Medium,Riesgo,0,0.25000,0.63158,0.633,0.090,0.28571,0.57895,0.13793,...,0.63158,0.633,0.090,0.28571,0.57895,0.13793,0.11111,0.75554,0.07,0.5596
1,High,Riesgo,1,0.79167,0.52632,0.462,0.354,0.71429,0.78947,0.37931,...,0.52632,0.462,0.354,0.71429,0.78947,0.37931,0.55556,0.60623,0.27,0.8222
2,Medium,Riesgo,0,0.58333,0.68421,0.499,0.006,0.71429,0.10526,0.62069,...,0.68421,0.499,0.006,0.71429,0.10526,0.62069,1.00000,0.43642,0.77,0.6177


In [36]:
# Etapa 7: Análisis Estadísticos
print('\n=== ETAPA 7: VERIFICACIONES ESTADÍSTICAS ===')

# Elimina los duplicados de columna si los hay
df_unique = df_final.loc[:, ~df_final.columns.duplicated()].copy()

# Crosstab: burnout_level por profile_type
if ('profile_type' in df_unique.columns) and ('burnout_level' in df_unique.columns):
    print('\nCrosstab: Proporción de burnout_level por profile_type')
    ct = pd.crosstab(df_unique['profile_type'], df_unique['burnout_level'], normalize='index')
    print(ct.round(3))
else:
    print('No se encontró profile_type o burnout_level')


=== ETAPA 7: VERIFICACIONES ESTADÍSTICAS ===

Crosstab: Proporción de burnout_level por profile_type
burnout_level  High    Low  Medium
profile_type                      
Riesgo         0.26  0.232   0.508


In [37]:
# Correlaciones Spearman con high_burnout
num_cols = df_unique.select_dtypes(include=[np.number]).columns.tolist()
if 'high_burnout' in df_unique.columns and num_cols:
    corr_list = []
    for c in num_cols:
        if c != 'high_burnout':
            try:
                r, p = spearmanr(df_unique[c], df_unique['high_burnout'], nan_policy='omit')
                corr_list.append((c, r, p))
            except:
                pass
    corr_df = pd.DataFrame(corr_list, columns=['variable', 'rho', 'pvalue'])
    corr_df = corr_df.sort_values('rho', key=abs, ascending=False)
    print('\nTop 10 correlaciones Spearman con high_burnout:')
    print(corr_df.head(10).to_string(index=False))


Top 10 correlaciones Spearman con high_burnout:
                    variable       rho        pvalue
    stress_level_scaled_norm  0.743002  0.000000e+00
           stress_level_norm  0.743002  0.000000e+00
       daily_work_hours_norm  0.437897  0.000000e+00
daily_work_hours_scaled_norm  0.437897  0.000000e+00
     screen_time_scaled_norm  0.405114 9.069926e-275
            screen_time_norm  0.405114 9.069926e-275
          sleep_quality_norm -0.398298 7.731844e-265
           bugs_per_day_norm  0.378533 2.466545e-237
    bugs_per_day_scaled_norm  0.378533 2.466545e-237
meetings_per_day_scaled_norm  0.243645  3.939921e-95


In [38]:
# ANOVA: sleep_hours por burnout_level
if 'sleep_hours' in df_clean.columns and 'burnout_level' in df_clean.columns:
    groups = [g.dropna().values for _, g in df_clean.groupby('burnout_level')['sleep_hours']]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) > 1:
        fstat, pval = f_oneway(*groups)
        print(f'\nANOVA: sleep_hours ~ burnout_level')
        print(f'F-statistic: {fstat:.3f}, p-value: {pval:.2e}')
        print('\nMedias por grupo:')
        means = df_clean.groupby('burnout_level')['sleep_hours'].agg(['count', 'mean', 'std'])
        print(means.round(3))


ANOVA: sleep_hours ~ burnout_level
F-statistic: 167.629, p-value: 8.38e-72

Medias por grupo:
               count   mean    std
burnout_level                     
High            1782  6.093  1.372
Low             1593  6.972  1.378
Medium          3485  6.462  1.417
